In [ ]:
!nvidia-smi

In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes trl -q

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from datasets import load_dataset

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
# Load famous Dolly dataset — 15k Q&A pairs
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

print(f"Total samples: {len(dataset)}")
print(f"\nSample entry:")
print(f"Category: {dataset[0]['category']}")
print(f"Instruction: {dataset[0]['instruction']}")
print(f"Response: {dataset[0]['response'][:200]}")

In [ ]:
def format_prompt(sample):
    """
    Convert each sample into instruction format.
    This is the exact format TinyLlama was trained on.
    """
    instruction = sample["instruction"]
    context     = sample.get("context", "")
    response    = sample["response"]

    if context:
        text = f"""<|system|>
You are a helpful assistant.</s>
<|user|>
{instruction}

Context: {context}</s>
<|assistant|>
{response}</s>"""
    else:
        text = f"""<|system|>
You are a helpful assistant.</s>
<|user|>
{instruction}</s>
<|assistant|>
{response}</s>"""

    return {"text": text}

# Apply formatting to entire dataset
dataset = dataset.map(format_prompt)

# Use only 1000 samples for fast training on free Colab
dataset = dataset.select(range(500))

print(f"Training samples: {len(dataset)}")
print(f"\nFormatted sample:")
print(dataset[0]["text"][:400])

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit quantization config — this is what makes QLoRA possible
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # load model in 4-bit
    bnb_4bit_quant_type="nf4",              # NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.float16,   # compute in float16
    bnb_4bit_use_double_quant=True,         # double quantization for extra savings
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def generate_response(prompt, max_new_tokens=200):
    """Generate a response from the model."""
    formatted = f"<|user|>\n{prompt}</s>\n<|assistant|>\n"
    inputs    = tokenizer(formatted, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("<|assistant|>")[-1].strip()

# Test before fine-tuning
print("=" * 50)
print("BEFORE FINE-TUNING:")
print("=" * 50)
print(generate_response("What is machine learning?"))

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=16,                          # rank — size of LoRA matrices
    lora_alpha=32,                 # scaling factor (usually 2x rank)
    target_modules=[               # which layers to apply LoRA to
        "q_proj",
        "v_proj",
        "k_proj",
        "o_proj",
    ],
    lora_dropout=0.05,             # dropout for regularization
    bias="none",
    task_type=TaskType.CAUSAL_LM,  # causal language modeling
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# See how many parameters we're actually training
model.print_trainable_parameters()
# Output will be something like:
# trainable params: 2,097,152 || all params: 1,102,048,256 || trainable%: 0.19
# Only 0.19% of parameters are being trained!

In [ ]:
training_args = TrainingArguments(
    output_dir="./tinyllama-finetuned",
    num_train_epochs=2,              # 2 passes through dataset
    per_device_train_batch_size=4,   # 4 samples per step
    gradient_accumulation_steps=4,   # accumulate gradients for 4 steps
    learning_rate=2e-4,              # learning rate
    fp16=True,                       # use float16 for speed
    logging_steps=50,                # print loss every 50 steps
    save_steps=200,                  # save checkpoint every 200 steps
    warmup_ratio=0.03,               # warmup for 3% of training
    lr_scheduler_type="cosine",      # cosine learning rate decay
    report_to="none",                # don't use wandb
)

print("✓ Training config ready")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="./tinyllama-finetuned",
        num_train_epochs=2,
        per_device_train_batch_size=1,      # ← reduced from 4 to 1
        gradient_accumulation_steps=8,      # ← increased to compensate
        learning_rate=2e-4,
        bf16=True,
        logging_steps=25,
        save_steps=200,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        report_to="none",
    ),
)

print("Starting training...")
trainer.train()
print("\n✓ Training complete!")

In [ ]:
print("=" * 50)
print("AFTER FINE-TUNING:")
print("=" * 50)
print(generate_response("What is machine learning?"))

print("\n" + "=" * 50)
print("More tests:")
print("=" * 50)
print("\nQ: What is supervised learning?")
print(generate_response("What is supervised learning?"))

print("\nQ: Explain neural networks simply")
print(generate_response("Explain neural networks simply"))

In [ ]:
model.save_pretrained("./tinyllama-lora-weights")
print("✓ Model saved!")

In [ ]:
from huggingface_hub import login

login()

In [ ]:
# Push LoRA weights to HuggingFace Hub
model.push_to_hub("tinyllama-dolly-finetuned")
tokenizer.push_to_hub("tinyllama-dolly-finetuned")

print("✓ Model pushed to HuggingFace Hub!")
print("  Visit: https://huggingface.co/Mehak-123-arora/tinyllama-dolly-finetuned")